<div style="display:flex; align-items:center; justify-content:space-between; gap:24px; border-bottom:1px solid #d0d7de; padding-bottom:12px; margin-bottom:18px;">
  <div>
    <h1 style="margin:0;">Infraestructuras Computacionales para el Procesamiento de datos masivos</h1>
    <p style="margin:6px 0 0 0;"><strong>Ejercicio 1, Modulo 3 Tema 1 - Consumidor de streaming con Kafka</strong></p>
  </div>
  <img src="https://www.uned.es/universidad/.imaging/mte/site-facultades-theme/220/dam/recursos-corporativos/logotipos/facultades-escuelas/logo_informatica.gif/jcr:content/logo_informatica.gif" alt="Logo ETSI Informatica UNED" style="height:72px; width:auto;" />
</div>

Este cuaderno consume eventos JSON desde un tópico de **Kafka** y calcula métricas con **Spark Structured Streaming**.

En esta versión, Kafka se utiliza a nivel básico como intermediario entre el productor y Spark:

```text
CoinGecko API
     ↓
Productor Python
     ↓
Kafka topic: crypto_prices
     ↓
Spark Structured Streaming
     ↓
Agregaciones por ventana
     ↓
Consola / Parquet
```

El objetivo no es profundizar todavía en Kafka, sino entender cómo Spark puede leer un flujo de eventos desde un sistema de mensajería.

## 1. Preparación del entorno

Ejecuta esta celda para crear las carpetas necesarias para la salida en Parquet y los checkpoints.

> Importante: ejecuta primero el cuaderno productor. Ese cuaderno arranca Kafka, crea el tópico `crypto_prices` y publica eventos en `localhost:9092`.

In [ ]:
import os
import shutil

OUTPUT_DIR = os.path.abspath("output/crypto_prices_parquet")
CHECKPOINT_DIR = os.path.abspath("checkpoint/crypto_prices_kafka")

# Limpieza para repetir el ejercicio desde cero.
# No se elimina ningún dato de Kafka; solo la salida y checkpoints de Spark.
for path in (OUTPUT_DIR, CHECKPOINT_DIR):
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)

print("Directorio de trabajo:", os.getcwd())
print("Carpetas preparadas y limpiadas:")
print("-", OUTPUT_DIR)
print("-", CHECKPOINT_DIR)

## 2. Instalación de PySpark

Si PySpark ya está instalado en tu entorno, puedes omitir esta celda.

Para leer desde Kafka, Spark necesita además el paquete `spark-sql-kafka-0-10`. Lo añadiremos al crear la `SparkSession`.

In [ ]:
# Ejecutar solo si PySpark no está instalado en el entorno
%pip install pyspark==3.5.1

## 3. Inicialización de SparkSession con soporte Kafka

Spark no incluye el conector Kafka en el paquete base de PySpark. Por eso se añade mediante `spark.jars.packages`.

La versión del conector debe coincidir con la versión de Spark usada en el entorno. En este ejemplo usamos Spark 3.5.1.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CoinGeckoKafkaStreaming")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

spark

## 4. Configuración básica de Kafka

En este ejercicio se usará un único tópico: `crypto_prices`.

Cada mensaje Kafka debe tener en el campo `value` un JSON con esta estructura:

```json
{
  "event_time": "2026-04-27T10:30:00Z",
  "coin": "bitcoin",
  "currency": "eur",
  "price": 58750.23
}
```

In [ ]:
KAFKA_BOOTSTRAP_SERVERS = "localhost:9092"
KAFKA_TOPIC = "crypto_prices"

print("Bootstrap servers:", KAFKA_BOOTSTRAP_SERVERS)
print("Topic:", KAFKA_TOPIC)

## 5. Definición del esquema de los eventos

Kafka entrega los mensajes como bytes. Spark leerá primero el `value` como texto y después lo transformará a columnas usando este esquema.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

schema = StructType([
    StructField("event_time", TimestampType(), True),
    StructField("coin", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("price", DoubleType(), True)
])

schema

## 6. Lectura del flujo desde Kafka

Spark leerá de Kafka mediante `readStream`.

Columnas importantes que entrega Kafka:

- `key`: clave del mensaje, si existe.
- `value`: contenido del mensaje.
- `topic`: tópico de origen.
- `partition`: partición Kafka.
- `offset`: posición del mensaje dentro de la partición.
- `timestamp`: marca temporal asignada por Kafka.

En este ejercicio nos centraremos en `value`, que contiene el JSON de CoinGecko.

In [ ]:
raw_kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

raw_kafka_df.printSchema()

## 7. Parseo del mensaje JSON

El campo `value` llega en binario, así que se convierte a `STRING` y después se aplica `from_json()`.

Además, conservamos metadatos básicos de Kafka como `topic`, `partition`, `offset` y `kafka_timestamp`.

In [ ]:
from pyspark.sql.functions import col, from_json

stream_df = (
    raw_kafka_df
    .select(
        col("topic"),
        col("partition"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp"),
        col("value").cast("string").alias("json_value")
    )
    .select(
        "topic",
        "partition",
        "offset",
        "kafka_timestamp",
        from_json(col("json_value"), schema).alias("data")
    )
    .select(
        "topic",
        "partition",
        "offset",
        "kafka_timestamp",
        col("data.event_time").alias("event_time"),
        col("data.coin").alias("coin"),
        col("data.currency").alias("currency"),
        col("data.price").alias("price")
    )
)

stream_df.printSchema()

## 8. Consulta básica en consola

Esta consulta permite comprobar que Spark está recibiendo mensajes desde Kafka.

Ejecuta primero el cuaderno productor y después esta celda. Con `startingOffsets="earliest"`, Spark puede leer también los eventos que ya estaban en el tópico antes de arrancar esta consulta.

In [ ]:
raw_query = (
    stream_df.writeStream
    .format("console")
    .option("checkpointLocation", os.path.join(CHECKPOINT_DIR, "raw_console"))
    .outputMode("append")
    .option("truncate", "false")
    .trigger(processingTime="10 seconds")
    .start()
)

raw_query.awaitTermination(60)
raw_query.stop()

## 9. Agregaciones por ventana temporal

Ahora se calculan métricas por criptomoneda en ventanas de 1 minuto:

- precio medio,
- precio máximo,
- precio mínimo,
- número de eventos procesados.

La marca de agua (`watermark`) permite gestionar eventos que lleguen tarde según el campo `event_time`.

In [ ]:
from pyspark.sql.functions import window, avg, max, min, count

metrics_df = (
    stream_df
    .withWatermark("event_time", "1 minute")
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("coin")
    )
    .agg(
        avg("price").alias("avg_price"),
        max("price").alias("max_price"),
        min("price").alias("min_price"),
        count("*").alias("num_events")
    )
)

metrics_df.printSchema()

## 10. Visualización de métricas en consola

La salida en modo `update` muestra las ventanas que se actualizan cuando llegan nuevos eventos.

De nuevo, la consulta se detiene automáticamente después de 60 segundos para que el ejercicio sea cómodo en Jupyter.

In [ ]:
metrics_query = (
    metrics_df.writeStream
    .option("checkpointLocation", os.path.join(CHECKPOINT_DIR, "metrics_console"))
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="10 seconds")
    .start()
)

metrics_query.awaitTermination(60)
metrics_query.stop()

## 11. Escritura de eventos en Parquet

Esta consulta guarda los eventos ya parseados en formato Parquet.

En streaming, Spark necesita una carpeta de checkpoint para recordar qué offsets de Kafka ya ha procesado.

In [ ]:
parquet_query = (
    stream_df.writeStream
    .format("parquet")
    .option("path", OUTPUT_DIR)
    .option("checkpointLocation", os.path.join(CHECKPOINT_DIR, "parquet"))
    .outputMode("append")
    .trigger(processingTime="10 seconds")
    .start()
)

parquet_query.awaitTermination(60)
parquet_query.stop()

## 12. Lectura posterior de los datos Parquet

Después de que la consulta anterior haya procesado algunos eventos, los datos almacenados se pueden leer como un DataFrame estático.

In [ ]:
historical_df = spark.read.parquet(OUTPUT_DIR)

historical_df.show(truncate=False)
historical_df.groupBy("coin").count().show()

## 13. Detener consultas activas

Es importante detener las consultas streaming cuando se termina el ejercicio para liberar recursos.

In [ ]:
for query in spark.streams.active:
    print("Deteniendo consulta:", query.name, query.id)
    query.stop()

print("Consultas activas:", spark.streams.active)

## 14. Cierre de SparkSession

Ejecuta esta celda al final del ejercicio.

In [ ]:
spark.stop()

## 15. Preguntas de autoevaluaci&oacute;n

1. Que diferencia hay entre leer ficheros JSON como streaming y leer eventos desde Kafka?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 1</strong></summary>

Leer JSON como streaming implica que Spark vigila una carpeta y procesa nuevos ficheros cuando aparecen. Leer desde Kafka implica consumir mensajes de un broker distribuido, con offsets, particiones, retencion y desacoplamiento entre productor y consumidor.

</details>

2. Que papel cumple Kafka entre el productor y Spark?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 2</strong></summary>

Kafka actua como sistema intermedio de mensajeria. Recibe eventos del productor, los almacena temporalmente en un topic y permite que Spark los consuma a su ritmo.

</details>

3. Por que Spark recibe el campo `value` como binario?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 3</strong></summary>

Spark recibe `value` como binario porque Kafka almacena claves y valores como bytes. La aplicacion consumidora debe convertir esos bytes a texto y despues interpretar el JSON con un esquema.

</details>

4. Para que sirven `partition` y `offset`?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 4</strong></summary>

`partition` indica la particion del topic Kafka donde esta almacenado el mensaje. `offset` identifica la posicion del mensaje dentro de esa particion. Juntos permiten saber exactamente que eventos se han leido.

</details>

5. Que significa `startingOffsets="latest"`?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 5</strong></summary>

`startingOffsets="latest"` significa que Spark empieza a consumir desde los mensajes nuevos que lleguen despues de iniciar la consulta, sin procesar mensajes antiguos ya existentes en el topic.

</details>

6. Que ocurriria si se usa `startingOffsets="earliest"`?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 6</strong></summary>

Con `startingOffsets="earliest"`, Spark intenta leer desde el primer offset disponible en Kafka. Esto permite reprocesar eventos antiguos si aun estan dentro del periodo de retencion del topic.

</details>

7. Por que el checkpoint es especialmente importante al consumir desde Kafka?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 7</strong></summary>

El checkpoint guarda el progreso de la consulta, incluidos offsets ya procesados y estado de agregaciones. Sin checkpoint, al reiniciar la aplicacion se puede perder continuidad o reprocesar eventos.

</details>

8. Que diferencia hay entre `event_time` y `kafka_timestamp`?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 8</strong></summary>

`event_time` es la marca temporal generada por el productor o por el evento de negocio. `kafka_timestamp` es la marca temporal asociada por Kafka al mensaje cuando se escribe o se registra en el broker.

</details>

9. Que ventajas tiene Kafka frente a una carpeta de ficheros como fuente streaming?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 9</strong></summary>

Kafka ofrece baja latencia, retencion configurable, particionado, tolerancia a fallos, multiples consumidores independientes y control exacto del progreso mediante offsets. Una carpeta de ficheros es mas simple, pero menos adecuada para flujos continuos de eventos.

</details>

10. Que limitaciones tiene este ejemplo respecto a una arquitectura de produccion?

<details>
<summary><strong>Ver soluci&oacute;n de la pregunta 10</strong></summary>

El ejemplo es didactico: usa un unico topic sencillo, no aplica seguridad, no gestiona errores avanzados, no contempla Schema Registry, no incluye monitorizacion productiva, no escala en cluster y no define una politica completa de despliegue y recuperacion.

</details>

## 16. Ejercicios de ampliaci&oacute;n

1. Cambiar `startingOffsets` de `latest` a `earliest` y observar el resultado.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 1</strong></summary>

En la lectura de Kafka, sustituye:

```python
.option("startingOffsets", "latest")
```

por:

```python
.option("startingOffsets", "earliest")
```

Si el topic ya contiene mensajes y Kafka aun los conserva, Spark los procesara desde el offset mas antiguo disponible. Para observar bien el cambio, conviene detener la consulta, limpiar el checkpoint o usar un checkpoint nuevo.

</details>

2. A&ntilde;adir una columna calculada con el precio redondeado a dos decimales.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 2</strong></summary>

```python
parsed_df = parsed_df.withColumn("price_rounded", F.round(F.col("price"), 2))
```

</details>

3. Filtrar solo los eventos de `bitcoin`.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 3</strong></summary>

```python
bitcoin_df = parsed_df.filter(F.col("coin") == "bitcoin")
```

</details>

4. Calcular ventanas de 5 minutos en lugar de 1 minuto.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 4</strong></summary>

```python
metrics_5m_df = (
    parsed_df
    .withWatermark("event_time", "10 minutes")
    .groupBy(F.window("event_time", "5 minutes"), "coin")
    .agg(
        F.avg("price").alias("avg_price"),
        F.min("price").alias("min_price"),
        F.max("price").alias("max_price"),
        F.count("*").alias("events")
    )
)
```

</details>

5. Guardar las m&eacute;tricas agregadas en Parquet.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 5</strong></summary>

```python
parquet_query = (
    metrics_5m_df
    .writeStream
    .format("parquet")
    .option("path", "output/kafka_metrics_parquet")
    .option("checkpointLocation", "checkpoint/kafka_metrics_parquet")
    .outputMode("append")
    .start()
)
```

Para usar `append` con ventanas es recomendable definir watermark. Si se usa una agregacion sin watermark, puede ser necesario usar `complete` con un sink compatible.

</details>

6. Crear una alerta cuando el precio de Bitcoin supere un umbral definido.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 6</strong></summary>

```python
BITCOIN_THRESHOLD = 70000

alerts_df = (
    parsed_df
    .filter(F.col("coin") == "bitcoin")
    .filter(F.col("price") > BITCOIN_THRESHOLD)
    .select("event_time", "kafka_timestamp", "coin", "price")
)
```

Se puede escribir en consola durante la practica:

```python
alerts_query = (
    alerts_df
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .start()
)
```

</details>

7. Comparar `event_time` con `kafka_timestamp`.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 7</strong></summary>

```python
latency_df = (
    parsed_df
    .withColumn(
        "latency_seconds",
        F.col("kafka_timestamp").cast("long") - F.col("event_time").cast("long")
    )
    .select("coin", "event_time", "kafka_timestamp", "latency_seconds")
)
```

Esta diferencia aproxima el retraso entre el tiempo del evento y el tiempo registrado por Kafka.

</details>

8. Modificar el productor para enviar la criptomoneda como `key` del mensaje Kafka.

<details>
<summary><strong>Ver soluci&oacute;n del ejercicio 8</strong></summary>

En el productor Kafka, envia la clave codificada en bytes:

```python
producer.send(
    "crypto_prices",
    key=event["coin"].encode("utf-8"),
    value=json.dumps(event).encode("utf-8")
)
```

En el consumidor Spark, convierte la clave a texto:

```python
kafka_df = kafka_df.withColumn("message_key", F.col("key").cast("string"))
```

Usar una clave permite que Kafka asigne de forma estable los mensajes de una misma moneda a la misma particion, siempre que el topic tenga varias particiones.

</details>
